# Initial configuration

In [ ]:
import os
import torch
from pathlib import Path
from matplotlib import font_manager
import matplotlib.pyplot as plt
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

QUICK_TEST = False
# Data Paths 
DATASET_PATH = '../DATA/etlcb/ETL9B'          # ETL9B Dataset (_unpack folders)
PREDICTION_FOLDER = '../TESTS'                       # Folder with images for prediction
CASIA_DATASET_TRAIN_PATH = "../DATA/chinese-handwriting/CASIA-HWDB_Train/Train" # CASIA Dataset Traububg Folder
CASIA_DATASET_TEST_PATH = "../DATA/chinese-handwriting/CASIA-HWDB_Test/Test" # CASIA Dataset Test Folder

# Output Paths
OUTPUT_DIR = './Training_Output'                     # Main output folder

# File Names
MODEL_FILENAME = 'best_kanji_model.pth'              # Trained model
HISTORY_FILENAME = 'training_history.json'           # Training history
CLASSES_FILENAME = 'classes.json'                    # Class mapping

# Full File Paths 
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, MODEL_FILENAME)
HISTORY_SAVE_PATH = os.path.join(OUTPUT_DIR, HISTORY_FILENAME)
CLASSES_SAVE_PATH = os.path.join(OUTPUT_DIR, CLASSES_FILENAME)
MODEL_PATH = MODEL_SAVE_PATH  # Alias for prediction scripts

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
if QUICK_TEST:
    MAX_CLASSES_LIMIT = 2
    BATCH_SIZE = 2
    NUM_EPOCHS = 1
    OPTUNA_TRIALS = 4
    OPTUNA_EPOCHS = 1
else:
    MAX_CLASSES_LIMIT = 150
    BATCH_SIZE = 128         
    NUM_EPOCHS = 30
    OPTUNA_TRIALS = 10
    OPTUNA_EPOCHS = 10
LEARNING_RATE = 0.0001
IMG_SIZE = 128

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Paths configuration loaded")
print(f"   • Dataset: {DATASET_PATH}")
print(f"   • Output: {OUTPUT_DIR}")
print(f"   • Model: {MODEL_SAVE_PATH}")

# Dataset and model setup

In [ ]:
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from tqdm import tqdm 
import matplotlib.pyplot as plt
import json
import pandas as pd
from PIL import Image
import os 
from torch.optim.lr_scheduler import ReduceLROnPlateau
import optuna
from optuna.trial import TrialState
from modules.dataset import ETL9Dataset
from modules.transforms import GaussianNoise, MorphologicalTransform
import modules.image_processing as ip

print(f"Indexing full dataset (Limit: {MAX_CLASSES_LIMIT})...")

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Data Augmentation and Transforms
train_transforms = transforms.Compose([ 
    transforms.Resize((IMG_SIZE, IMG_SIZE)), 
    transforms.Grayscale(num_output_channels=3), # MobileNet requires 3 channels, so we replicate the grayscale channel
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3), 
    transforms.RandomAffine( 
        degrees=10,
        translate=(0.05, 0.05),
        scale=(0.9, 1.1),
        shear=5
    ),
    transforms.ElasticTransform(alpha=10.0, sigma=5.0),     
    transforms.ColorJitter(brightness=0.3, contrast=0.3),   
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), 
    MorphologicalTransform(kernel_size=3, p=0.5),
    transforms.ToTensor(),
    GaussianNoise(0., 0.03), 
    transforms.RandomErasing(p=0.3), 
    ip.NORM
])

val_transforms = transforms.Compose([ 
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    ip.NORM
])

# Load and Index Dataset
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Root directory not found at: {DATASET_PATH}")

print("Indexing full dataset...")
# Aquí aplicamos el límite definido al inicio de la celda
full_dataset = ETL9Dataset(root_dir=DATASET_PATH, transform=None, max_classes=MAX_CLASSES_LIMIT)

if len(full_dataset) == 0:
    raise ValueError("Dataset is empty. Check path and unpacked folders.")

# Split dataset into Train, Validation, and Test
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size
print(f"Total: {total_size}")
print(f"Split: Train={train_size}, Val={val_size}, Test={test_size}")


# Create random indices for splitting
generator = torch.Generator().manual_seed(42)
indices = torch.randperm(total_size, generator=generator).tolist()
train_idx = indices[:train_size]
val_idx = indices[train_size : train_size + val_size]
test_idx = indices[train_size + val_size :]

print("Preparing subsets...")
train_dataset_augmented = ETL9Dataset(root_dir=DATASET_PATH, transform=train_transforms, max_classes=MAX_CLASSES_LIMIT) # Should be here after data split
base_dataset = ETL9Dataset(root_dir=DATASET_PATH, transform=val_transforms, max_classes=MAX_CLASSES_LIMIT) # Should be here after data split

train_subset = Subset(train_dataset_augmented, train_idx)
val_subset = Subset(base_dataset, val_idx)
test_subset = Subset(base_dataset, test_idx)


def get_dataloaders_reduced(batch_size):
    samples_per_class = 50
    
    filtered_indexes = []
    
    for class_name in full_dataset.classes:
        class_samples_indexes = [
            i for i, sample in enumerate(full_dataset.samples) 
            if sample[1] == class_name
        ]
        selected_indexes = class_samples_indexes[:samples_per_class]
        filtered_indexes.extend(selected_indexes)
    
    subset_optuna = Subset(train_dataset_augmented, filtered_indexes)
    train_size_opt = int(0.8 * len(subset_optuna))
    val_size_opt = len(subset_optuna) - train_size_opt
    
    train_opt, val_opt = torch.utils.data.random_split(
        subset_optuna, [train_size_opt, val_size_opt]
    )

    return (
        DataLoader(train_opt, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True),
        DataLoader(val_opt, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
        len(full_dataset.classes)
    )


def get_dataloaders(batch_size):
    """Initialize DataLoaders with dynamic batch_size"""
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader

def build_model(num_classes):
    """Model Setup"""
    model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
    input_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(input_features, num_classes) # Adapting last layer
    return model.to(device)

# Initialize Defaults loaders
train_loader, val_loader = get_dataloaders(BATCH_SIZE)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print("DataLoaders ready")

model = build_model(len(full_dataset.classes))
print(f"{type(model).__name__} configured for {len(full_dataset.classes)} classes.")

# Dataset distribution

In [ ]:
from collections import Counter
class_counts = Counter([sample[1] for sample in full_dataset.samples])
unique_counts = set(class_counts.values())
print(f"All classes have same number of images: {len(unique_counts) == 1}")
print(f"Range: min={min(unique_counts)}, max={max(unique_counts)}")

# Optuna hyperparameter optimization

In [ ]:
from modules.optuna import objective

study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())

study.optimize(
    lambda trial: objective(
        trial, 
        get_dataloaders_fn=get_dataloaders_reduced, 
        build_model_fn=build_model, 
        device=device, 
        optuna_epochs=OPTUNA_EPOCHS
    ), 
    n_trials=OPTUNA_TRIALS
)

optuna.visualization.plot_optimization_history(study).show()
optuna.visualization.plot_param_importances(study).show()
optuna.visualization.plot_parallel_coordinate(study).show()

print("\n--- Optimization finished ---")
print(f"Best Accuracy: {study.best_value:.4f}")
print(f"Best parameters: {study.best_params}")

# Model training

In [ ]:
import json
from modules.train_utils import EarlyStopping
from modules.train_model import train_model

RESUME_TRAINING = False # Set true to resume training from last checkpoint
if RESUME_TRAINING and os.path.exists(os.path.join(OUTPUT_DIR, 'last_checkpoint.pth')):
    checkpoint = torch.load(os.path.join(OUTPUT_DIR, 'last_checkpoint.pth'))
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    history = checkpoint['history']
    best_acc = checkpoint['best_acc']
    print(f"Restarting from epoch {start_epoch}")
else:
    start_epoch = 0
    history = None
    best_acc = 0.0

# Execution with Optuna computed parameters
# Getting best parameters
best_params = study.best_params
print(f"Best params: {best_params}")

# Getting models and dataloaders
train_loader, val_loader = get_dataloaders(best_params['batch_size'])
test_loader = DataLoader(test_subset, batch_size=best_params['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
model = build_model(len(full_dataset.classes))

criterion = nn.CrossEntropyLoss() # Loss function
optimizer = optim.AdamW(
    model.parameters(), 
    lr=best_params['lr'], 
    weight_decay=best_params['weight_decay']
)  

scheduler = ReduceLROnPlateau(
    optimizer, 
    mode='max',        # Max val_accuracy
    factor=0.5,        # Decrease LR to half it's value
    patience=2,        
    min_lr=1e-6       # Minimum value for LR
)

early_stopping = EarlyStopping(patience=5)

# Training
model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS,
    output_dir=OUTPUT_DIR,
    model_save_path=MODEL_SAVE_PATH,
    device=device,
    early_stopping=early_stopping,
    scheduler=scheduler,
    start_epoch=start_epoch,
    best_acc=best_acc,
    history=history
)

# Save history and classes
with open(HISTORY_SAVE_PATH, 'w') as f:
    json.dump(history, f)
print(f"History saved to: {HISTORY_SAVE_PATH}")

with open(CLASSES_SAVE_PATH, 'w') as f:
    json.dump(full_dataset.classes, f, ensure_ascii=False)  
print(f"Classes saved to: {CLASSES_SAVE_PATH}")

# Training metrics

In [ ]:
import json
import matplotlib.pyplot as plt

def plot_training_history():
    with open(HISTORY_SAVE_PATH, 'r') as f:
        history = json.load(f)
    
    acc = history['train_acc']
    val_acc = history['val_acc']
    loss = history['train_loss']
    val_loss = history['val_loss']
    epochs_range = range(1, len(acc) + 1)
    
    plt.figure(figsize=(14, 5))
    
    # Accuracy plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy', color='blue')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy', color='orange')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend(loc='lower right')
    plt.grid(True)
    
    # Loss plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss', color='blue')
    plt.plot(epochs_range, val_loss, label='Validation Loss', color='orange')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend(loc='upper right')
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
plot_training_history()

# Test evaluation

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import torch
from tqdm import tqdm
import json
from modules import fonts
import modules.image_processing as ip

# Loading a displayable japanese font
fonts.load_font()

# Function to display some tests
def visualize_errors_and_metrics(model, test_loader, class_names):
    model.eval() # Set model to evaluation mode
    all_preds = []
    all_labels = []
    mistakes = [] 
    MISTAKES_TO_DISPLAY = 20
    ROWS = 4
    COLUMNS = 5
    
    print("Analyzing Test data...")
    
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Processing"):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            conf, preds = torch.max(probs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            wrong_indices = (preds != labels).nonzero()
            for idx in wrong_indices:
                idx = idx.item()
                if len(mistakes) < MISTAKES_TO_DISPLAY: # Just displaying limited mistakes
                    img_tensor = inputs[idx]
                    img = ip.denormalize(img_tensor)
                    
                    mistakes.append({
                        'img': img,
                        'pred_lbl': class_names[preds[idx].item()],
                        'true_lbl': class_names[labels[idx].item()],
                        'conf': conf[idx].item()
                    })

    # Displaying mistakes
    if len(mistakes) > 0:
        fig, axes = plt.subplots(ROWS, COLUMNS, figsize=(15, 12))
        fig.suptitle('Error Gallery (Actual -> Predicted [Confidence])', fontsize=16)
        axes = axes.flatten()
        
        for i, ax in enumerate(axes):
            if i < len(mistakes):
                m = mistakes[i]
                ax.imshow(m['img'])
                
                color = 'red' if m['conf'] > 0.8 else 'orange'
                ax.set_title(f"A: {m['true_lbl']}\nP: {m['pred_lbl']}\nConf: {m['conf']:.2f}", 
                             color=color, fontsize=10, fontweight='bold')
            ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print("No errors found in the visualized sample.")

    # Stats
    df = pd.DataFrame({
        'Actual': [class_names[i] for i in all_labels],
        'Predicted': [class_names[i] for i in all_preds]
    })
    
    errors_df = df[df['Actual'] != df['Predicted']]
    
    if not errors_df.empty:
        print("\n--- TOP KANJIS WITH MOST ERRORS ---")
        print(errors_df['Actual'].value_counts().head(10))
        
        print("\n--- MOST COMMON CONFUSIONS (Actual -> Predicted) ---")
        print(errors_df.groupby(['Actual', 'Predicted']).size().sort_values(ascending=False).head(10))
    
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    print(f"\nGlobal Test Accuracy: {acc*100:.2f}%")

# Execution
try:
    with open(CLASSES_SAVE_PATH, 'r') as f:
        class_names = json.load(f)
    print(f"Classes loaded from {CLASSES_SAVE_PATH} ({len(class_names)} classes).")
except FileNotFoundError:
    print(f"File not found {CLASSES_SAVE_PATH}. Using global variable 'full_dataset'...")
    class_names = full_dataset.classes

model = model.to(device)
visualize_errors_and_metrics(model, test_loader, class_names)

# Loading model for external evaluation

In [ ]:
import torch
from modules.evaluation import predict_and_evaluate
from modules.fonts import load_font

load_font()
model = build_model(len(class_names)) 
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model = model.to(device)

# Evaluation with samples that aren't in the dataset

In [ ]:
predict_and_evaluate(
    model=model,
    folder_path=PREDICTION_FOLDER,
    class_names=class_names,
    device=device,
    img_size=IMG_SIZE,
    skip_not_in_classes=False,
    display=True
)

# Evaluation with other dataset (CASIA training set)

In [ ]:
print("====== Evaluation with CASIA train set ======")
predict_and_evaluate(
    model=model,
    folder_path=CASIA_DATASET_TRAIN_PATH,
    class_names=class_names,
    device=device,
    img_size=IMG_SIZE,
    skip_not_in_classes=True,
    display=False
)
print("====== Evaluation with CASIA test set ======")
predict_and_evaluate(
    model=model,
    folder_path=CASIA_DATASET_TEST_PATH,
    class_names=class_names,
    device=device,
    img_size=IMG_SIZE,
    skip_not_in_classes=True,
    display=False
)

# Evaluation with samples that aren't in the dataset MC Dropout

In [ ]:
predict_and_evaluate(
    model=model,
    folder_path=PREDICTION_FOLDER,
    class_names=class_names,
    device=device,
    img_size=IMG_SIZE,
    skip_not_in_classes=False,
    display=True,
    mc_iterations=10
)

# Evaluation with other dataset (CASIA training set) MC Dropout

In [ ]:
print("====== Evaluation with CASIA train set ======")
predict_and_evaluate(
    model=model,
    folder_path=CASIA_DATASET_TRAIN_PATH,
    class_names=class_names,
    device=device,
    img_size=IMG_SIZE,
    skip_not_in_classes=True,
    display=False,
    mc_iterations=10
)
print("====== Evaluation with CASIA test set ======")
predict_and_evaluate(
    model=model,
    folder_path=CASIA_DATASET_TEST_PATH,
    class_names=class_names,
    device=device,
    img_size=IMG_SIZE,
    skip_not_in_classes=True,
    display=False,
    mc_iterations=10
)